# Framing Annotation — Step 2: Test Annotation Instructions Against Human Labels

We merge the two manual label files, run the LLM on the same paragraphs, and compare.
Step 2.1 runs at temperature 0 to check basic coverage.
Step 2.2 runs multiple times at higher temperature to surface hard cases.

## Setup

In [1]:
import pandas as pd
import requests
import os
import time
from collections import Counter
from dotenv import load_dotenv
from sklearn.metrics import classification_report, confusion_matrix

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

print(f"API key loaded: {'YES' if api_key else 'NO'}")

API key loaded: YES


## Load and Merge Manual Labels

In [2]:
df1 = pd.read_csv("manual-label-1.csv")
df2 = pd.read_csv("manual-label-2.csv")

df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(subset=["text_block", "label"])
df = df[df["text_block"].str.split().str.len() >= 15].reset_index(drop=True)

# normalize human labels to uppercase
df["human_label"] = df["label"].str.strip().str.upper()

print(f"Total rows after merging and filtering: {len(df)}")
print("\nHuman label distribution:")
print(df["human_label"].value_counts())

Total rows after merging and filtering: 400

Human label distribution:
human_label
NOT_SECURITY_CRIME    326
SECURITY_NEGATIVE      74
Name: count, dtype: int64


## Annotation Instructions

Edit these between rounds based on what you find in the inspection cells below.

In [3]:
instructions = """Sie sind ein neutraler Annotationsassistent fuer ein wissenschaftliches Projekt zur Inhaltsanalyse deutscher Zeitungsartikel.

Ihre Aufgabe ist es, jeden Zeitungsabsatz als SECURITY_NEGATIVE oder NOT_SECURITY_CRIME zu klassifizieren.

SECURITY_NEGATIVE: Der Absatz rahmt eine Gruppe, Massnahme oder ein Ereignis explizit als Gefahr, Bedrohung, Kriminalitaetsquelle, Instabilitaetsfaktor, Gewaltrisiko oder oeffentliches Sicherheitsproblem. Die Gruppe wird als Ursache oder Quelle des Problems dargestellt.

NOT_SECURITY_CRIME: Alles andere. Vergeben Sie dieses Label wenn:
- Kein Sicherheits- oder Kriminalitaetsrahmen vorhanden ist
- Die genannte Gruppe das OPFER von Gewalt, Kriminalitaet, Verfolgung oder staatlicher Repression ist
- Kriminalitaet auf strukturelle Ursachen wie Armut oder soziale Ausgrenzung zurueckgefuehrt wird, nicht auf die Gruppe selbst
- Der Absatz einen Sicherheitsrahmen explizit in Frage stellt oder widerlegt
- Es sich um internationale militaerische Konflikte zwischen Staaten handelt (z.B. Russland-Ukraine-Krieg, NATO-Einsaetze, Waffenlieferungen zwischen Laendern) - dieses Projekt untersucht nur die innenpolitische Rahmung von Minderheiten- oder Migrationsgruppen in Deutschland
- Es sich um historische Gewalt handelt, bei der die genannte Gruppe das Opfer war (z.B. Holocaust, Pogrome, koloniale Gewalt)
- Neutrale Erwaehnung von Polizei, Gerichten oder Grenzen ohne klare Bedrohungszuschreibung an die Gruppe"""

reminder = "Klassifizieren Sie den Absatz als SECURITY_NEGATIVE oder NOT_SECURITY_CRIME. Nur diese zwei Labels sind erlaubt. SECURITY_NEGATIVE nur wenn die Gruppe explizit als Bedrohung oder Kriminalitaetsquelle dargestellt wird und nicht Opfer ist."

print("Instructions loaded.")

Instructions loaded.


## Core Functions

In [4]:
def annotate(text, instructions, reminder, temperature=0.0):
    prompt = (
        f"{instructions}\n\n"
        f"Annotieren Sie jetzt diesen Absatz:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Label: <SECURITY_NEGATIVE oder NOT_SECURITY_CRIME>\n"
        "Explanation: <ein Satz der Ihre Entscheidung begruendet>"
    )
    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": [
            {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Befolgen Sie die Annotationsanweisungen und antworten Sie immer im angegebenen Format."},
            {"role": "user",   "content": prompt}
        ]
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type":  "application/json"
    }
    for attempt in range(3):
        try:
            r = requests.post(f"{HOST}/chat/completions", json=payload, headers=headers, timeout=120)
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"].strip()
            print(f"  [attempt {attempt+1}] status {r.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] error: {e}")
            time.sleep(5)
    return "ERROR"


def parse_label(raw):
    u = raw.upper()
    if "NOT_SECURITY_CRIME" in u or "NOT SECURITY CRIME" in u:
        return "NOT_SECURITY_CRIME"
    elif "SECURITY_NEGATIVE" in u or "SECURITY NEGATIVE" in u:
        return "SECURITY_NEGATIVE"
    return "UNCLEAR"


def parse_explanation(raw):
    for line in raw.split("\n"):
        if line.lower().startswith("explanation:"):
            return line[len("explanation:"):].strip()
    return raw


print("Functions loaded.")

Functions loaded.


## Step 2.1 — Annotate All Human-Labelled Rows at Temperature 0

We run the LLM on every row in your merged human-labelled dataset and compare against your gold standard labels.
After running, inspect the results below — look at what it got wrong and why.

In [5]:
OUTPUT_2_1 = "results/step2_1_results.csv"
os.makedirs("results", exist_ok=True)

results_2_1 = []

for i, row in df.iterrows():
    raw         = annotate(str(row["text_block"]), instructions, reminder, temperature=0.0)
    llm_label   = parse_label(raw)
    explanation = parse_explanation(raw)

    results_2_1.append({
        "group":       row["group"],
        "pub":         row["pub"],
        "human_label": row["human_label"],
        "llm_label":   llm_label,
        "explanation": explanation,
        "text_block":  row["text_block"],
        "raw_output":  raw,
    })

    if (i + 1) % 20 == 0:
        done = pd.DataFrame(results_2_1)
        correct = (done["human_label"] == done["llm_label"]).sum()
        unclear = (done["llm_label"] == "UNCLEAR").sum()
        print(f"  [{i+1}/{len(df)}]  correct: {correct}  unclear: {unclear}")

annotation_2_1 = pd.DataFrame(results_2_1)
annotation_2_1.to_csv(OUTPUT_2_1, index=False)
print(f"\nDone. Saved to {OUTPUT_2_1}")

  [20/400]  correct: 14  unclear: 0
  [40/400]  correct: 29  unclear: 0
  [60/400]  correct: 47  unclear: 0
  [80/400]  correct: 63  unclear: 0
  [100/400]  correct: 77  unclear: 0
  [120/400]  correct: 95  unclear: 0
  [140/400]  correct: 113  unclear: 0
  [160/400]  correct: 129  unclear: 0
  [180/400]  correct: 138  unclear: 0
  [200/400]  correct: 142  unclear: 0
  [220/400]  correct: 161  unclear: 0
  [240/400]  correct: 181  unclear: 0
  [260/400]  correct: 200  unclear: 0
  [280/400]  correct: 220  unclear: 0
  [300/400]  correct: 240  unclear: 0
  [320/400]  correct: 259  unclear: 0
  [340/400]  correct: 279  unclear: 0
  [360/400]  correct: 299  unclear: 0
  [380/400]  correct: 318  unclear: 0
  [400/400]  correct: 337  unclear: 0

Done. Saved to results/step2_1_results.csv


### 2.1 — Label Distribution and UNCLEAR Check

In [6]:
total = len(annotation_2_1)
unclear_pct = (annotation_2_1["llm_label"] == "UNCLEAR").sum() / total * 100

print("LLM label distribution:")
print(annotation_2_1["llm_label"].value_counts())
print(f"\nUNCLEAR: {unclear_pct:.1f}%")
if unclear_pct > 5:
    print("WARNING: more than 5% unclear — adjust the reminder or output format instructions")

LLM label distribution:
llm_label
NOT_SECURITY_CRIME    369
SECURITY_NEGATIVE      31
Name: count, dtype: int64

UNCLEAR: 0.0%


### 2.1 — F1, Precision, Recall Against Human Labels

In [7]:
# exclude UNCLEAR rows from metrics
eval_df = annotation_2_1[annotation_2_1["llm_label"] != "UNCLEAR"].copy()

print(f"Rows used for evaluation (excluding UNCLEAR): {len(eval_df)} of {total}")
print()
print(classification_report(
    eval_df["human_label"],
    eval_df["llm_label"],
    labels=["SECURITY_NEGATIVE", "NOT_SECURITY_CRIME"]
))

print("Confusion matrix (rows=human, cols=LLM):")
cm = confusion_matrix(
    eval_df["human_label"],
    eval_df["llm_label"],
    labels=["SECURITY_NEGATIVE", "NOT_SECURITY_CRIME"]
)
cm_df = pd.DataFrame(
    cm,
    index=["human: SEC_NEG", "human: NOT_SEC"],
    columns=["llm: SEC_NEG", "llm: NOT_SEC"]
)
print(cm_df)

Rows used for evaluation (excluding UNCLEAR): 400 of 400

                    precision    recall  f1-score   support

 SECURITY_NEGATIVE       0.68      0.28      0.40        74
NOT_SECURITY_CRIME       0.86      0.97      0.91       326

          accuracy                           0.84       400
         macro avg       0.77      0.63      0.65       400
      weighted avg       0.82      0.84      0.82       400

Confusion matrix (rows=human, cols=LLM):
                llm: SEC_NEG  llm: NOT_SEC
human: SEC_NEG            21            53
human: NOT_SEC            10           316


### 2.1 — Inspect Errors

Look at what the LLM got wrong. These are your candidates for adding decision rules or few-shot examples.

In [8]:
errors = annotation_2_1[
    (annotation_2_1["llm_label"] != annotation_2_1["human_label"]) &
    (annotation_2_1["llm_label"] != "UNCLEAR")
].copy()

print(f"Errors: {len(errors)} out of {total} ({len(errors)/total*100:.1f}%)")
print("\nError breakdown:")
print(errors.groupby(["human_label", "llm_label"]).size())

Errors: 63 out of 400 (15.8%)

Error breakdown:
human_label         llm_label         
NOT_SECURITY_CRIME  SECURITY_NEGATIVE     10
SECURITY_NEGATIVE   NOT_SECURITY_CRIME    53
dtype: int64


In [9]:
# false positives: LLM said SECURITY_NEGATIVE but human said NOT_SECURITY_CRIME
fp = errors[errors["llm_label"] == "SECURITY_NEGATIVE"]
print(f"False positives (LLM overcalls threat framing): {len(fp)}")
print()
for _, row in fp.iterrows():
    print(f"Group: {row['group']} | Pub: {row['pub']}")
    print(f"LLM explanation: {row['explanation']}")
    print(f"Text: {str(row['text_block'])[:300]}")
    print("TODO: Why did it get this wrong? Add a decision rule.")
    print("-" * 60)

False positives (LLM overcalls threat framing): 10

Group: ASYL | Pub: LVZ
LLM explanation: Der Absatz rahmt die steigende Zahl der Schutzsuchenden (insbesondere aus Syrien, Afghanistan, der Türkei und Venezuela) als **Belastung für die Kapazitäten und öffentliche Ordnung** („wir steuern auf eine Lage zu, wie wir sie schon hatten“), impliziert eine **Instabilität** („Hoch-Zeiten“) und stellt die Gruppe indirekt als Ursache für Überlastung und Verteilungsprobleme dar.
Text: Drei Viertel der [Andere Gruppe] stammen aus Syrien

Zum Vergleich führte der Innenminister an: Bis Mai kamen pro Monat rund 500 [Andere Gruppe] nach Sachsen, im Juli stieg diese Zahl bereits auf 1500 und im August auf 2000. " Die Prognosen sagen, dass das kein kurzfristiger Effekt ist" , warnte Sch
TODO: Why did it get this wrong? Add a decision rule.
------------------------------------------------------------
Group: KURD | Pub: KN
LLM explanation: Der Absatz stellt die **[Gruppe 1] (PYD)** explizit als mögliche **

In [10]:
# false negatives: LLM said NOT_SECURITY_CRIME but human said SECURITY_NEGATIVE
fn = errors[errors["llm_label"] == "NOT_SECURITY_CRIME"]
print(f"False negatives (LLM misses threat framing): {len(fn)}")
print()
for _, row in fn.iterrows():
    print(f"Group: {row['group']} | Pub: {row['pub']}")
    print(f"LLM explanation: {row['explanation']}")
    print(f"Text: {str(row['text_block'])[:300]}")
    print("TODO: Why did it miss this? Add a decision rule or few-shot example.")
    print("-" * 60)

False negatives (LLM misses threat framing): 53

Group: BGR | Pub: Bild
LLM explanation: Der Absatz beschreibt zwar statistische Daten zu Tätern, stellt aber keine Gruppe explizit als Bedrohung oder Kriminalitätsquelle dar, sondern nennt nur Herkunftsangaben ohne wertende Zuschreibung.
Text: [[ Zwei Gruppenvergewaltiger waren unter 14, zwei weitere über 70. Größte Gruppe: 14-30 (179 Personen). Auch hier gab's weibliche Tatverdächtige: 14 von 221. Besonders schwere Fälle begingen 58 Personen.

[[ Die Herkunft der Tatverdächtigen: Weit vorne bei Vergewaltigungen und Gruppenvergewaltigunge
TODO: Why did it miss this? Add a decision rule or few-shot example.
------------------------------------------------------------
Group: ASYL | Pub: LVZ
LLM explanation: Der Absatz beschreibt eine Einzelhandlung eines Tatverdächtigen aus der Gruppe, ohne die Gruppe selbst als Ursache oder Quelle der Bedrohung oder Kriminalität zu rahmen.
Text: 

Wegen einer Bedrohungssituation sind am Silvestertag zwei 

### Notes from 2.1 — fill in after inspection

| Round | Error Type | Problem | Decision Rule Added |
|-------|-----------|---------|--------------------|
| 1     | FP        |         |                    |
| 1     | FN        |         |                    |
| 2     | FP        |         |                    |

## Step 2.2 — Hard Cases: Multiple Runs at Higher Temperature

We take the rows where the LLM made errors in 2.1 plus a random sample of correct cases, run each 5 times at temperature 0.7, and look for disagreement across runs.
Cases with low agreement are hard cases where the instructions leave room for interpretation.

In [11]:
N_RUNS          = 5
TEMPERATURE_2_2 = 0.7
N_CORRECT_SAMPLE = 20
OUTPUT_2_2      = "results/step2_2_hardcases.csv"
RANDOM_SEED     = 42

# use all error rows plus a random sample of correct rows
correct_sample = annotation_2_1[
    annotation_2_1["llm_label"] == annotation_2_1["human_label"]
].sample(n=min(N_CORRECT_SAMPLE, len(annotation_2_1)), random_state=RANDOM_SEED)

hard_case_pool = pd.concat([errors, correct_sample], ignore_index=True)
hard_case_pool = hard_case_pool.drop_duplicates(subset="text_block").reset_index(drop=True)

print(f"Running {N_RUNS} passes on {len(hard_case_pool)} rows ({len(errors)} errors + {len(correct_sample)} correct sample)")

Running 5 passes on 83 rows (63 errors + 20 correct sample)


In [12]:
results_2_2 = []

for i, row in hard_case_pool.iterrows():
    run_labels = []
    for r in range(N_RUNS):
        raw = annotate(str(row["text_block"]), instructions, reminder, temperature=TEMPERATURE_2_2)
        run_labels.append(parse_label(raw))

    counts      = Counter(run_labels)
    final_label = counts.most_common(1)[0][0]
    agreement   = counts.most_common(1)[0][1] / N_RUNS

    entry = {
        "group":       row["group"],
        "pub":         row["pub"],
        "human_label": row["human_label"],
        "final_label": final_label,
        "agreement":   agreement,
        "text_block":  row["text_block"],
    }
    for r in range(N_RUNS):
        entry[f"run_{r+1}"] = run_labels[r]

    results_2_2.append(entry)
    print(f"  row {i+1}/{len(hard_case_pool)} — {final_label} (agreement {agreement:.0%}) | human: {row['human_label']}")

annotation_2_2 = pd.DataFrame(results_2_2).sort_values("agreement").reset_index(drop=True)
annotation_2_2.to_csv(OUTPUT_2_2, index=False)
print(f"\nDone. Saved to {OUTPUT_2_2}")

  row 1/83 — NOT_SECURITY_CRIME (agreement 80%) | human: SECURITY_NEGATIVE
  row 2/83 — NOT_SECURITY_CRIME (agreement 80%) | human: SECURITY_NEGATIVE
  row 3/83 — NOT_SECURITY_CRIME (agreement 100%) | human: SECURITY_NEGATIVE
  row 4/83 — NOT_SECURITY_CRIME (agreement 100%) | human: SECURITY_NEGATIVE
  row 5/83 — NOT_SECURITY_CRIME (agreement 100%) | human: SECURITY_NEGATIVE
  row 6/83 — NOT_SECURITY_CRIME (agreement 80%) | human: SECURITY_NEGATIVE
  row 7/83 — NOT_SECURITY_CRIME (agreement 100%) | human: SECURITY_NEGATIVE
  row 8/83 — NOT_SECURITY_CRIME (agreement 100%) | human: SECURITY_NEGATIVE
  row 9/83 — NOT_SECURITY_CRIME (agreement 80%) | human: SECURITY_NEGATIVE
  row 10/83 — NOT_SECURITY_CRIME (agreement 100%) | human: SECURITY_NEGATIVE
  row 11/83 — NOT_SECURITY_CRIME (agreement 100%) | human: SECURITY_NEGATIVE
  row 12/83 — NOT_SECURITY_CRIME (agreement 100%) | human: SECURITY_NEGATIVE
  row 13/83 — NOT_SECURITY_CRIME (agreement 100%) | human: SECURITY_NEGATIVE
  row 14/83 

### 2.2 — Agreement Distribution

In [13]:
run_cols = [f"run_{r+1}" for r in range(N_RUNS)]

print("Agreement distribution:")
print(annotation_2_2["agreement"].describe())
print(f"\nFull agreement (100%):  {(annotation_2_2['agreement'] == 1.0).sum()} rows")
print(f"Hard cases (<80%):      {(annotation_2_2['agreement'] < 0.8).sum()} rows")
print(f"Very hard (<60%):       {(annotation_2_2['agreement'] < 0.6).sum()} rows")

Agreement distribution:
count    83.000000
mean      0.968675
std       0.085437
min       0.600000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: agreement, dtype: float64

Full agreement (100%):  72 rows
Hard cases (<80%):      2 rows
Very hard (<60%):       0 rows


In [14]:
# inspect hard cases sorted by lowest agreement first
hard = annotation_2_2[annotation_2_2["agreement"] < 0.8]
print(f"Hard cases (agreement < 80%): {len(hard)}")

for _, row in hard.iterrows():
    runs = [row[f"run_{r+1}"] for r in range(N_RUNS)]
    print(f"\n{'='*60}")
    print(f"Agreement: {row['agreement']:.0%}  |  Final: {row['final_label']}  |  Human: {row['human_label']}")
    print(f"Runs: {runs}")
    print(f"Group: {row['group']} | Pub: {row['pub']}")
    print(f"Text: {str(row['text_block'])[:400]}")
    print("TODO: Why is this hard? What decision rule would resolve it?")

Hard cases (agreement < 80%): 2

Agreement: 60%  |  Final: SECURITY_NEGATIVE  |  Human: NOT_SECURITY_CRIME
Runs: ['NOT_SECURITY_CRIME', 'NOT_SECURITY_CRIME', 'SECURITY_NEGATIVE', 'SECURITY_NEGATIVE', 'SECURITY_NEGATIVE']
Group: ASYL | Pub: WAZ
Text: Solange Menschen den Drogenrausch brauchen, müssen Polizisten im Kampf gegen Rauschgiftkriminalität dicke Bretter bohren . Wie eng Erfolg und Frust zusammenliegen, zeigt ein Vorfall am Willy-Brandt-Platz, wo der Zivilfahnder einen mutmaßlichen westafrikanischen Drogendealer im Visier hat. Bei der Festnahme bringt er den Mann zu Boden, doch dieser widersetzt sich hartnäckig. Nach Darstellung des Er
TODO: Why is this hard? What decision rule would resolve it?

Agreement: 60%  |  Final: NOT_SECURITY_CRIME  |  Human: NOT_SECURITY_CRIME
Runs: ['NOT_SECURITY_CRIME', 'NOT_SECURITY_CRIME', 'SECURITY_NEGATIVE', 'SECURITY_NEGATIVE', 'NOT_SECURITY_CRIME']
Group: EGY | Pub: OSZ
Text: Der Anführer des Terrornetzwerks Al-Kaida, Aiman al-Sawahiri, ist bei

### Notes from 2.2 — fill in after inspection

| Round | Agreement | Human Label | LLM runs | Problem | Decision Rule Added |
|-------|-----------|-------------|----------|---------|--------------------|
| 1     |           |             |          |         |                    |
| 2     |           |             |          |         |                    |

## Summary: When to Stop

You are done with Step 2 when:
- UNCLEAR labels are below 5%
- F1 for SECURITY_NEGATIVE is above 0.7
- Hard cases (agreement < 80%) are below 20% of your test set

If not there yet: update the instructions cell above, add decision rules based on your notes, and rerun from Step 2.1.

Once done, move to Step 3: few-shot examples and chain-of-thought prompting on a representative sample.